In [ ]:
label_mapping = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4,
}

train["label"] = train["answer"].map(label_mapping)

In [ ]:
print(train.loc[150, "answer"])
print(train.loc[150, "label"])

In [ ]:
formatted_input = (
    str(train.loc[0, "prompt"])
    + " [SEP] "
    + str(train.loc[0, "B"])
)

In [ ]:
print(len(formatted_input))

In [ ]:
from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, TaskType, get_peft_model

model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, lora_config)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(trainable_params)

In [ ]:
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

# -------------------------------------------------------
# Load Dataset
# -------------------------------------------------------

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv").iloc[:32].copy()

label_mapping = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4,
}

train["label"] = train["answer"].map(label_mapping)

# -------------------------------------------------------
# Tokenizer
# -------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# -------------------------------------------------------
# Preprocessing
# -------------------------------------------------------

def preprocess(example):

    choices = [
        str(example["A"]),
        str(example["B"]),
        str(example["C"]),
        str(example["D"]),
        str(example["E"]),
    ]

    tokenized = tokenizer(
        [example["prompt"]] * 5,
        choices,
        padding="max_length",
        truncation=True,
        max_length=64,
    )

    return {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "labels": example["label"],
    }

dataset = Dataset.from_pandas(train)
dataset = dataset.map(preprocess)

dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)

# -------------------------------------------------------
# Load Model
# -------------------------------------------------------

model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

# -------------------------------------------------------
# Apply LoRA
# -------------------------------------------------------

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, lora_config)

print("\nTrainable Parameters")
model.print_trainable_parameters()

# -------------------------------------------------------
# Trainer
# -------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./outputs",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# -------------------------------------------------------
# Train
# -------------------------------------------------------

trainer.train()

print("\nQ9 Answer")
print("Global Step =", trainer.state.global_step)

# -------------------------------------------------------
# Inference on Row 0
# -------------------------------------------------------

row = train.iloc[0]

choices = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"]),
]

encoding = tokenizer(
    [row["prompt"]] * 5,
    choices,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt",
)

input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

trainer.model.eval()

with torch.no_grad():

    outputs = trainer.model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

probabilities = torch.softmax(
    outputs.logits,
    dim=1,
)

print("\nProbabilities")
print(probabilities)

print("\nQ10 Answer")
print(
    "Probability of Option E =",
    round(probabilities[0, 4].item(), 4),
)